# CSCI 5622 HW #4 - Questions (a) and (b)

In [54]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import BertTokenizer, BertModel
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import os

## Data processing

In [15]:
labels_path = r"../Study 1 (EDAIC)/DepressionLabels.csv"
transcripts_path = r"../Study 1 (EDAIC)/EDAIC Transcripts"

# Read depression labels
labels_df = pd.read_csv(labels_path)
labels_df

,Participant_ID,PHQ_Score
0,300,2
1,301,3
2,302,4
3,303,0
4,304,6
...,...,...
214,698,19
215,702,0
216,703,8
217,707,1


### Identify participants for which data is available

In [38]:
# Check that labels provided have a matching transcript
# Disregard participant data where a label/score or transcript are unavailable
ids = list()
transcripts_list = list()
for i in labels_df['Participant_ID'].to_numpy():
    if os.path.isfile(f'{transcripts_path}/{str(i)}_Transcript.csv'):
        ids.append(i)
        transcripts_list.append(f'{str(i)}_Transcript.csv')
print(len(ids))

134


### Gather language features for each transcript

In [62]:
def read_transcript_csv(path):
    with open(path, 'r') as f:
        lines = f.readlines()
    processed_lines = [l.strip() for l in lines[1:]]
    return processed_lines

def compute_transcript_sentiment(lines:list):
    '''
    Code written using Microsoft Copilot
    Calculates average of each sentiment score as given by the Vader sentiment package
    '''
    analyzer = SentimentIntensityAnalyzer()
    scores = [analyzer.polarity_scores(statement) for statement in lines]
    
    # Aggregate by mean
    avg_compound = sum(s['compound'] for s in scores) / len(scores)
    avg_pos = sum(s['pos'] for s in scores) / len(scores)
    avg_neg = sum(s['neg'] for s in scores) / len(scores)
    avg_neu = sum(s['neu'] for s in scores) / len(scores)
    
    return {
        'avg_compound': avg_compound,
        'avg_pos': avg_pos,
        'avg_neg': avg_neg,
        'avg_neu': avg_neu
    }

def get_transcript_embeddings(lines:list):
    '''
    Code written using Microsoft Copilot
    Get average embedding of a transcript using pretrained BERT by mean pooling all statement embeddings 
    '''
    # Define BERT model
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased')

    embeddings = list()
    for statement in lines: # Get embedding for each statement
        inputs = tokenizer(statement, return_tensors='pt', truncation=True, padding=True)
        with torch.no_grad():
            outputs = model(**inputs)
        # Get [CLS] token embedding (first token)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # shape: [1, hidden_size]
        embeddings.append(cls_embedding)  

    # Calculate average embedding across all statements
    transcript_embedding = torch.mean(torch.cat(embeddings, dim=0), dim=0)
    return transcript_embedding

In [57]:
# Example: list of transcripts, each transcript is a list of statements
transcripts = [
    ["I love this product!", "It works really well.", "Amazing experience."],
    ["This was okay.", "Not great, not terrible.", "Could be better."]
]

# Apply to all transcripts
transcript_scores = [compute_transcript_sentiment(t) for t in transcripts]

print(transcript_scores)


[{'avg_compound': 0.5313, 'avg_pos': 0.612, 'avg_neg': 0.0, 'avg_neu': 0.38800000000000007}, {'avg_compound': 0.000733333333333345, 'avg_pos': 0.35966666666666663, 'avg_neg': 0.24366666666666667, 'avg_neu': 0.39666666666666667}]


In [51]:
p = os.path.join(transcripts_path, '386_Transcript.csv')
l = read_transcript_csv(p)
print(l)

['might have pulled something that', "I'm going to bring the great thanks so much", 'and please', 'are you okay with this yes', "oh I'm fine I'm a little tired but I found out my thyroid is I think acting up so", 'where are you from originally I was born in Canada', "but I've lived in California most of my life so and it's gray today too so the gray weather makes you kind of see no sluggish", 'oh my gosh years and years ago', "not at all I don't think I can text to her for a long time well the air fares have gone up quite a bit in the last few years gas prices have gone up so travel as much as it's a fun thing to do it's cost-prohibitive so", "oh that's see everything has plus and minuses cold-weather not as much sun so I think I'd prefer getting more sun and having the warm-weather so but other than that it's it's very pretty back their natural scenery in but I like it here you know the traffic is kind of heavy but that's a small thing to deal with", 'why did I move to LA because my f

In [64]:
# Iterate through all relevant CSV files
for file in transcripts_list:
    ls = read_transcript_csv(os.path.join(transcripts_path, file))
    
    # Get average sentiment scores using Vader
    sentiment_scores = compute_transcript_sentiment(ls)
    
    # Extract average embedding for each transcript using pretrained BERT
    embedding = get_transcript_embeddings(ls)
    print('Testing: transcript done')

Testing: transcript done
Testing: transcript done
Testing: transcript done
Testing: transcript done


KeyboardInterrupt: 